# MSigDB ORA Saturation Analysis (study subsampling)

**Environment:** `clamp-analyses`

For each CLAMPfull model in the study-level saturation grid (`08_saturation_study`, varying K across study coverage levels and up to 3 seeds), this notebook:

1. Loads the Z matrix (gene loadings per LV).
2. For each LV, selects the top 1% genes by descending loading as the gene list.
3. Runs `enricher()` per LV using MSigDB (v2026.1) as gene set database and model genes as universe.
4. Stores raw `terms_padj`: the minimum p.adjust per MSigDB term across all LVs (no FDR threshold applied here).
5. Saves per-model RDS caches (`_msigdb.rds`) and a summary CSV. FDR thresholds are applied in `01_bp_saturation_study_plot.ipynb`.

In [36]:
library(here)
library(dplyr)
library(clusterProfiler)
library(parallel)

## Paths

In [37]:
models_dir <- here("output/01_model_building/04_archs4/08_saturation_study")
output_dir <- here("output/03_model_biology/00_archs4/01_pathway_coverage_bp_saturation_study/00_bp_saturation_study_ora_analysis")

dir.create(file.path(output_dir, "CLAMPfull"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(output_dir, "CLAMPbase"), recursive = TRUE, showWarnings = FALSE)

## Load MSigDB gene sets

In [38]:
msig_gmt <- clusterProfiler::read.gmt(here("data/pathways/msigdb.v2026.1.Hs.symbols.gmt"))
message(sprintf("MSigDB gene sets loaded: %d", length(unique(msig_gmt$term))))

MSigDB gene sets loaded: 35361



## Model grid: discovered dynamically from disk

Scans `08_saturation_study` for all `study_saturation_rs{pct}_k{k}_seed_{seed}` directories that have a `CLAMPfull_hall/Z.csv`. K values and study coverage levels are not hardcoded.

In [39]:
all_subdirs <- list.dirs(models_dir, recursive = FALSE, full.names = FALSE)

model_grid <- do.call(rbind, lapply(all_subdirs, function(d) {
  m <- regmatches(d, regexec("^study_saturation_rs([0-9]+)_k([0-9]+)_seed_([0-9]+)$", d))[[1]]
  if (length(m) < 4) return(NULL)
  z_path_full <- file.path(models_dir, d, "CLAMPfull_hall", "Z.csv")
  if (!file.exists(z_path_full)) return(NULL)
  data.frame(
    rs_pct        = as.integer(m[2]),
    k_val         = as.integer(m[3]),
    seed          = as.integer(m[4]),
    subdir        = d,
    z_path_full   = z_path_full,
    z_path_base   = file.path(models_dir, d, "CLAMPbase", "Z.csv"),
    rds_path_full = file.path(models_dir, d, "CLAMPfull_hall.rds"),
    rds_path_base = file.path(models_dir, d, "CLAMPbase.rds"),
    model_dir     = file.path(models_dir, d),
    stringsAsFactors = FALSE
  )
}))

if (is.null(model_grid) || nrow(model_grid) == 0) {
  stop("No saturation models found in: ", models_dir)
}

model_grid <- model_grid[order(model_grid$rs_pct, model_grid$k_val, model_grid$seed), ]
rownames(model_grid) <- NULL

message("Available models: ", nrow(model_grid))
print(model_grid[, c("rs_pct", "k_val", "seed")])

Available models: 57



   rs_pct k_val seed
1       1    86    1
2       1    86    2
3       1    86    3
4       1   173    1
5       1   173    2
6       1   173    3
7       1   432    1
8       1   432    2
9       1   432    3
10      1   864    1
11      1   864    2
12      1   864    3
13      1  1296    1
14      1  1296    2
15      1  1296    3
16      1  1728    2
17      1  1728    3
18      5    86    1
19      5    86    2
20      5    86    3
21      5   173    1
22      5   173    2
23      5   173    3
24      5   432    1
25      5   432    2
26      5   432    3
27      5   864    1
28      5   864    2
29      5   864    3
30      5  1296    1
31      5  1296    2
32      5  1296    3
33      5  1728    1
34      5  1728    2
35      5  1728    3
36     10    86    1
37     10    86    2
38     10    86    3
39     10   173    1
40     10   173    2
41     10   173    3
42     10   432    1
43     10   432    2
44     10   432    3
45     10   864    1
46     10   864    2
47     10   8

## Helper: get n_studies from subsample_info

`subsample_info.rds` lives in the upstream `07_bp_coverage_study` source dirs, not in the model dir itself.

In [40]:
study_base_dir <- here("output/01_model_building/04_archs4/07_bp_coverage_study")
study_dirs     <- list.dirs(study_base_dir, recursive = FALSE, full.names = FALSE)

pct_to_subdir <- list()
for (d in study_dirs) {
  m <- regmatches(d, regexec("([0-9]+)$", d))[[1]]
  if (length(m) >= 2) pct_to_subdir[[as.character(as.integer(m[2]))]] <- d
}

get_n_studies <- function(rs_pct, seed_idx) {
  subdir <- pct_to_subdir[[as.character(rs_pct)]]
  if (is.null(subdir)) return(NA_integer_)
  src_path <- file.path(
    study_base_dir, subdir,
    sprintf("study_coverage_rs%d_seed_%d", rs_pct, seed_idx),
    "subsample_info.rds"
  )
  if (file.exists(src_path)) readRDS(src_path)$n_studies else NA_integer_
}

## Helper: run ORA for one model

Returns a list with `terms_padj` (minimum p.adjust per MSigDB term across all LVs, raw, no FDR cutoff). LVs are processed serially within each worker so outer `mclapply` parallelism is safe (no nested forking).

In [41]:
run_ora_for_model <- function(z_path, rds_path) {
  Z              <- read.csv(z_path, row.names = 1, check.names = FALSE)
  universe_genes <- rownames(Z)
  n_top          <- ceiling(0.01 * nrow(Z))
  n_lvs          <- ncol(Z)

  term_overlap   <- tapply(msig_gmt$gene %in% universe_genes, msig_gmt$term, sum)
  n_total_msigdb <- sum(term_overlap >= 10L)

  top_genes_per_lv <- apply(Z, 2, function(lv) {
    universe_genes[order(lv, decreasing = TRUE)[seq_len(n_top)]]
  })

  n_samples <- NA_integer_
  if (file.exists(rds_path)) {
    rds_obj <- readRDS(rds_path)
    if (!is.null(rds_obj$B)) n_samples <- ncol(rds_obj$B)
    rm(rds_obj)
  }

  ora_results <- lapply(seq_len(n_lvs), function(i) {
    genes <- top_genes_per_lv[, i]
    tryCatch(
      clusterProfiler::enricher(
        gene          = genes,
        universe      = universe_genes,
        TERM2GENE     = msig_gmt,
        pAdjustMethod = "BH",
        pvalueCutoff  = 1,
        qvalueCutoff  = 1,
        minGSSize     = 10,
        maxGSSize     = 50000
      ),
      error = function(e) NULL
    )
  })

  all_dfs <- Filter(Negate(is.null), lapply(ora_results, function(r) {
    if (is.null(r) || nrow(as.data.frame(r)) == 0) return(NULL)
    as.data.frame(r)
  }))

  if (length(all_dfs) == 0) {
    warning("No ORA results returned for: ", z_path)
    return(NULL)
  }

  combined <- do.call(rbind, all_dfs)

  list(
    n_samples      = n_samples,
    n_lvs          = n_lvs,
    n_top_genes    = n_top,
    n_total_msigdb = n_total_msigdb,
    terms_padj     = tapply(combined$p.adjust, combined$ID, min)
  )
}

## Helper: build results row from ORA output

In [42]:
build_row <- function(spec, res, n_studies) {
  data.frame(
    rs_pct         = spec$rs_pct,
    k_val          = spec$k_val,
    seed           = spec$seed,
    n_studies      = n_studies,
    n_samples      = res$n_samples,
    n_lvs          = res$n_lvs,
    n_top_genes    = res$n_top_genes,
    n_total_msigdb = res$n_total_msigdb,
    stringsAsFactors = FALSE
  )
}

## Run ORA: CLAMPfull and CLAMPbase (parallelised over models)

In [ ]:
n_workers <- 3L
run_seeds <- 1L  # set to NULL to run all seeds

show_pending <- function(model_subdir, seeds = NULL) {
  idx <- if (is.null(seeds)) seq_len(nrow(model_grid)) else which(model_grid$seed %in% seeds)
  cache_paths <- file.path(
    output_dir, model_subdir,
    sprintf("rs%d_k%d_seed%d_msigdb.rds",
            model_grid$rs_pct[idx], model_grid$k_val[idx], model_grid$seed[idx])
  )
  cached  <- idx[ file.exists(cache_paths)]
  missing <- idx[!file.exists(cache_paths)]
  seed_lbl <- if (is.null(seeds)) "all seeds" else paste0("seed", paste(seeds, collapse = ","))
  message(sprintf("%s [%s]: %d cached, %d to run",
                  model_subdir, seed_lbl, length(cached), length(missing)))
  for (i in missing) {
    message(sprintf("  TO RUN: rs%d%% k%d seed%d",
                    model_grid$rs_pct[i], model_grid$k_val[i], model_grid$seed[i]))
  }
}

show_pending("CLAMPfull", run_seeds)
show_pending("CLAMPbase", run_seeds)

run_one <- function(i, model_subdir, z_col, rds_col) {
  spec       <- model_grid[i, ]
  n_studies  <- get_n_studies(spec$rs_pct, spec$seed)
  cache_path <- file.path(
    output_dir, model_subdir,
    sprintf("rs%d_k%d_seed%d_msigdb.rds", spec$rs_pct, spec$k_val, spec$seed)
  )
  label <- sprintf("rs%d%% k%d seed%d [%d/%d]",
                   spec$rs_pct, spec$k_val, spec$seed, i, nrow(model_grid))

  if (file.exists(cache_path)) {
    res <- readRDS(cache_path)
    if (is.null(res$n_studies)) {
      res$n_studies <- n_studies
      saveRDS(res, cache_path)
    }
    message(sprintf("[CACHED]  %s", label))
  } else {
    t0 <- proc.time()[[3]]
    message(sprintf("[START]   %s (%d LVs)", label, spec$k_val))
    res <- run_ora_for_model(spec[[z_col]], spec[[rds_col]])
    elapsed <- round(proc.time()[[3]] - t0)
    if (!is.null(res)) {
      res$n_studies <- n_studies
      saveRDS(res, cache_path)
      message(sprintf("[DONE]    %s in %ds -> saved", label, elapsed))
    } else {
      message(sprintf("[FAILED]  %s after %ds", label, elapsed))
    }
  }

  if (is.null(res)) return(NULL)
  build_row(spec, res, n_studies)
}

run_idx  <- if (is.null(run_seeds)) seq_len(nrow(model_grid)) else which(model_grid$seed %in% run_seeds)
seed_lbl <- if (is.null(run_seeds)) "all seeds" else paste0("seed", paste(run_seeds, collapse = ","))

message(sprintf("\nRunning CLAMPfull ORA (%d workers, %s)...", n_workers, seed_lbl))
results_list_full <- parallel::mclapply(
  run_idx,
  run_one,
  model_subdir   = "CLAMPfull",
  z_col          = "z_path_full",
  rds_col        = "rds_path_full",
  mc.cores       = n_workers,
  mc.preschedule = FALSE
)

n_ok_full   <- sum(!sapply(results_list_full, is.null))
n_fail_full <- length(run_idx) - n_ok_full
message(sprintf("CLAMPfull done: %d/%d succeeded, %d failed/NULL",
                n_ok_full, length(run_idx), n_fail_full))

results_full_df <- do.call(rbind, Filter(Negate(is.null), results_list_full))
rownames(results_full_df) <- NULL
results_full_df <- results_full_df %>% dplyr::arrange(rs_pct, k_val, seed)

In [ ]:
for (pct in sort(unique(results_full_df$rs_pct))) {
  pct_df   <- results_full_df[results_full_df$rs_pct == pct, ]
  rds_path <- file.path(output_dir, "CLAMPfull", sprintf("results_pct%d_msigdb.rds", pct))
  csv_path <- file.path(output_dir, "CLAMPfull", sprintf("results_pct%d_msigdb.csv", pct))
  saveRDS(pct_df, rds_path)
  write.csv(pct_df, csv_path, row.names = FALSE)
  message(sprintf("Saved CLAMPfull pct%d: %d models", pct, nrow(pct_df)))
}

Saved CLAMPfull pct1: 8 models

Saved CLAMPfull pct5: 6 models

Saved CLAMPfull pct10: 8 models

Saved CLAMPfull pct25: 3 models



In [ ]:
message(sprintf("\nRunning CLAMPbase ORA (%d workers, %s)...", n_workers, seed_lbl))
results_list_base <- parallel::mclapply(
  run_idx,
  run_one,
  model_subdir   = "CLAMPbase",
  z_col          = "z_path_base",
  rds_col        = "rds_path_base",
  mc.cores       = n_workers,
  mc.preschedule = FALSE
)

n_ok_base   <- sum(!sapply(results_list_base, is.null))
n_fail_base <- length(run_idx) - n_ok_base
message(sprintf("CLAMPbase done: %d/%d succeeded, %d failed/NULL",
                n_ok_base, length(run_idx), n_fail_base))

results_base_df <- do.call(rbind, Filter(Negate(is.null), results_list_base))
rownames(results_base_df) <- NULL
results_base_df <- results_base_df %>% dplyr::arrange(rs_pct, k_val, seed)

In [ ]:
for (pct in sort(unique(results_base_df$rs_pct))) {
  pct_df   <- results_base_df[results_base_df$rs_pct == pct, ]
  rds_path <- file.path(output_dir, "CLAMPbase", sprintf("results_pct%d_msigdb.rds", pct))
  csv_path <- file.path(output_dir, "CLAMPbase", sprintf("results_pct%d_msigdb.csv", pct))
  saveRDS(pct_df, rds_path)
  write.csv(pct_df, csv_path, row.names = FALSE)
  message(sprintf("Saved CLAMPbase pct%d: %d models", pct, nrow(pct_df)))
}